In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.listdir('/content/drive/MyDrive/vaaniResearch/malayalam_speech_ai')

['ml', 'experiment']

In [ ]:
DATA_DIR = '/content/drive/MyDrive/vaaniResearch/malayalam_speech_ai/ml'

print(os.path.exists(DATA_DIR))
print(os.listdir(DATA_DIR)[:15])

True
['clip_durations.tsv', 'test.tsv', 'other.tsv', 'validated.tsv', 'invalidated.tsv', 'train.tsv', 'dev.tsv', 'reported.tsv', 'validated_sentences.tsv', 'unvalidated_sentences.tsv', 'README.md', '.DS_Store', 'clips']


In [ ]:
import pandas as pd

test_df = pd.read_csv(
    os.path.join(DATA_DIR, 'test.tsv'),
    sep='\t'
)

print("Number of rows:", len(test_df))
print("Columns:", test_df.columns.tolist())

display(test_df.head(5))

Number of rows: 894
Columns: ['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']


,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,5340d0912c2c74963f25056bd1fedcf51cbf8337b6bc5d...,common_voice_ml_37658248.mp3,cc28e5d7710f0589de14d88608b268d5ea118ac6d7e309...,ആ ശുഭരാത്രിയിലേയ്ക് ശാന്തമായി പോകരുത്,NaN,2,0,twenties,NaN,i speak like how it's written. south central k...,NaN,ml,NaN
1,5340d0912c2c74963f25056bd1fedcf51cbf8337b6bc5d...,common_voice_ml_37867190.mp3,0150e277bcb467bdbc9774116cf06ecc4c5c940eab5d4e...,"അതായത്, നമ്മുടെ മനസ്സിൽ നിലനിൽക്കുന്ന ആശയങ്ങളു...",NaN,4,0,twenties,NaN,i speak like how it's written. south central k...,NaN,ml,NaN
2,5340d0912c2c74963f25056bd1fedcf51cbf8337b6bc5d...,common_voice_ml_37859500.mp3,e892ba51a0b325500a72199c8b45e5f74579439fd637f4...,നമ്മളില് ഏറ്റവും നല്ലവനെന്ന്,NaN,4,0,twenties,NaN,i speak like how it's written. south central k...,NaN,ml,NaN
3,5340d0912c2c74963f25056bd1fedcf51cbf8337b6bc5d...,common_voice_ml_37859492.mp3,d3514c0e764b001e27a944d754abc59c1b0ba3122f5dac...,അമ്മാ ആഭരങ്ങളെടുക്കാൻ ആരാണ് പോവുന്നത്,NaN,2,0,twenties,NaN,i speak like how it's written. south central k...,NaN,ml,NaN
4,5340d0912c2c74963f25056bd1fedcf51cbf8337b6bc5d...,common_voice_ml_37859491.mp3,d983339aada14459d86012267052465ada69c50a47418f...,നാം അവിടെ എത്താൻ ആഗ്രഹിക്കുന്നു,NaN,2,0,twenties,NaN,i speak like how it's written. south central k...,NaN,ml,NaN


In [ ]:
from pathlib import Path

CLIPS_DIR = Path(DATA_DIR) / 'clips'

test_df['audio_path'] = test_df['path'].apply(
    lambda p: CLIPS_DIR / p
)

test_df['audio_exists'] = test_df['audio_path'].apply(
    lambda p: p.is_file()
)

print("Total test rows:", len(test_df))
print("Audio files found:", test_df['audio_exists'].sum())
print("Audio files missing:", (~test_df['audio_exists']).sum())

Total test rows: 894
Audio files found: 894
Audio files missing: 0


In [ ]:
import pandas as pd
from pathlib import Path

# Keep only rows with usable paths and reference text.
eligible = test_df.copy()

eligible["sentence"] = eligible["sentence"].fillna("").astype(str).str.strip()
eligible = eligible[
    (eligible["sentence"] != "") &
    eligible["audio_exists"]
].copy()

# Remove duplicate audio paths, if any.
eligible = eligible.drop_duplicates(subset=["path"])

print("Eligible recordings:", len(eligible))
print("Unique speakers:", eligible["client_id"].nunique())
print("Unique reference sentences:", eligible["sentence_id"].nunique())

# Select a reproducible random sample.
SEED = 42

evaluation_df = eligible.sample(
    n=100,
    random_state=SEED
).sort_values("path").reset_index(drop=True)

print("Selected examples:", len(evaluation_df))
display(evaluation_df[[
    "path", "sentence", "client_id", "sentence_id"
]].head(10))

Eligible recordings: 894
Unique speakers: 93
Unique reference sentences: 894
Selected examples: 100


,path,sentence,client_id,sentence_id
0,common_voice_ml_30346653.mp3,വൈറസിനെ ജനങ്ങൾക്ക് ഇടയിലേക്ക് ഇറക്കി വിട്ട് ഹെ...,c8e015354b9788b35ad07ff41dfdb0bf638e489ce2fa14...,790526fc7a6cf61f87735a0e7e66ebbb28b9a92a3fd000...
1,common_voice_ml_30346678.mp3,നമുക്കിവിടെ എന്തെല്ലാം ഉണ്ട്,c8e015354b9788b35ad07ff41dfdb0bf638e489ce2fa14...,789f7aa2e061517806cfaf55907e8452c406409a2104e0...
2,common_voice_ml_30523994.mp3,ഞങ്ങൾ വധുവിനെ ആശീര്വദിക്കാൻ വന്നതാണ്,36f42eee0768af1751bde02736f445b2f006a3e979454b...,6e3f43b28f58ae25f475ce5fdd2cb0f9454fb9d6cda99f...
3,common_voice_ml_30533768.mp3,അവനെ തടയ് പിടിക്ക്,36f42eee0768af1751bde02736f445b2f006a3e979454b...,51ff6b8794c4aa80de325b0948312d8c1fe742b8091ac3...
4,common_voice_ml_30533790.mp3,വേഗം അതെനിക്ക് തരൂ,36f42eee0768af1751bde02736f445b2f006a3e979454b...,78a5878ee0d4b1a502ef30db5024e56810eac4ec6d82aa...
5,common_voice_ml_30602494.mp3,അവനൊരു പൊട്ടിയ യന്ത്രം കണ്ടു,dd72354c03ab84af23953f82fe5108b17f2d9fbb0d5bfb...,7acbb14ea1262ef8c0cde8d5ab830ff737b9025419b9b7...
6,common_voice_ml_32261041.mp3,എന്നിട്ട് വേണം രാജ്യത്ത് പേരുകള് നിരോധിച്ച് പക...,82e6e487a5bf6fd5901946fc6e05ec523f0bc9fb574fbd...,841288b9f3f92e95d61acf4dea7641c946411f811b984a...
7,common_voice_ml_32261077.mp3,ഇന്ദിരാവധത്തിനെ തുടര്ന്നുണ്ടായ സിഖ് വിരുദ്ധ കല...,82e6e487a5bf6fd5901946fc6e05ec523f0bc9fb574fbd...,86a14b42c71b848274fd959a367270bfccc2afda8194f4...
8,common_voice_ml_32261353.mp3,മുഖചിത്രം മൾട്ടി കളറിൽ വേണം എന്ന ആശയം ഉയർന്നു ...,85abd23ac6c76e33ec4706358abbb75669b601726ca5d5...,723a52eecc9fd40688f0993aa4878658906bca6936862d...
9,common_voice_ml_32261480.mp3,ആര്ക്കും ഗുരുതരമായ രോഗ ലക്ഷണങ്ങളില്ല.,85abd23ac6c76e33ec4706358abbb75669b601726ca5d5...,87ffb1a052d4fd87b3132611a399f2393dd8bb1e968b1d...


In [ ]:
evaluation_df["audio_path"] = evaluation_df["path"].apply(
    lambda p: str(CLIPS_DIR / p)
)

manifest_columns = [
    "path",
    "audio_path",
    "sentence",
    "client_id",
    "sentence_id",
    "up_votes",
    "down_votes",
]

manifest = evaluation_df[manifest_columns].copy()

OUTPUT_DIR = Path("/content/drive/MyDrive/vaaniResearch/malayalam_speech_ai/experiment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = OUTPUT_DIR / "evaluation_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("Saved:", manifest_path)
print("Number of examples:", len(manifest))

Saved: /content/drive/MyDrive/vaaniResearch/malayalam_speech_ai/experiment/evaluation_manifest.csv
Number of examples: 100


In [ ]:
import random
import librosa
from IPython.display import Audio, display

row = evaluation_df.iloc[0]

audio, sr = librosa.load(
    row["audio_path"],
    sr=16000,
    mono=True
)

print("Reference transcript:")
print(row["sentence"])

print("Audio duration:", round(len(audio) / sr, 2), "seconds")
display(Audio(audio, rate=sr))

Reference transcript:
വൈറസിനെ ജനങ്ങൾക്ക് ഇടയിലേക്ക് ഇറക്കി വിട്ട് ഹെർഡ് ഇമ്മ്യൂണിറ്റി നേടിയടുക്കാം
Audio duration: 6.34 seconds


ASR

In [ ]:
!pip -q install transformers accelerate librosa jiwer soundfile

In [ ]:
import torch
import transformers
import librosa
import jiwer

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — check the Colab runtime settings.")

PyTorch version: 2.11.0+cu130
Transformers version: 5.20.2
CUDA available: True
GPU: Tesla T4


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline

MODEL_ID = "sajilck/whisper-small-malayalam"

# Load the audio processor and tokenizer configuration
processor = AutoProcessor.from_pretrained(MODEL_ID)

# Load the pretrained speech-recognition model
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16
).to("cuda")

# Create the transcription pipeline
asr = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=0,
    torch_dtype=torch.float16,
)

print("Model loaded:", MODEL_ID)
print("Device:", next(model.parameters()).device)
print("Model parameter dtype:", next(model.parameters()).dtype)

processor_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/315 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.85k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Model loaded: sajilck/whisper-small-malayalam
Device: cuda:0
Model parameter dtype: torch.float16


In [ ]:
import pandas as pd
from pathlib import Path
import time

MANIFEST_PATH = (
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/experiment/evaluation_manifest.csv"
)

evaluation_df = pd.read_csv(MANIFEST_PATH)

# Select the first fixed evaluation example
sample = evaluation_df.iloc[0]
audio_path = Path(sample["audio_path"])
reference = str(sample["sentence"])

# Transcribe in Malayalam
start = time.perf_counter()

prediction = asr(
    str(audio_path),
    generate_kwargs={
        "language": "malayalam",
        "task": "transcribe",
        "max_new_tokens": 444,
    }
)

elapsed_seconds = time.perf_counter() - start

print("Reference :", reference)
print("Prediction:", prediction["text"].strip())
print(f"Elapsed time: {elapsed_seconds:.2f} seconds")

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Reference : വൈറസിനെ ജനങ്ങൾക്ക് ഇടയിലേക്ക് ഇറക്കി വിട്ട് ഹെർഡ് ഇമ്മ്യൂണിറ്റി നേടിയടുക്കാം
Prediction: വൈറസിനെ ജനങ്ങൾക്കിടിയിലേക്ക് ഇറക്കിവിട്ട് ഹെയ്ഡ് ഇമ്മ്യൂണിറ്റി നേടിയെടുക്കാം.
Elapsed time: 23.26 seconds


In [ ]:
import time
import librosa
import torch

# Measure the actual duration of the sample
audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
duration = len(audio) / sr

print(f"Audio duration: {duration:.2f} seconds")
print(f"Sample rate: {sr} Hz")
print(f"First inference time: {elapsed_seconds:.2f} seconds")
print(f"First-run real-time factor: {elapsed_seconds / duration:.2f}")

# Repeat the same inference to investigate warm-up effects
if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.perf_counter()

prediction_repeat = asr(
    str(audio_path),
    generate_kwargs={
        "language": "malayalam",
        "task": "transcribe",
        "max_new_tokens": 444,
    }
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

repeat_seconds = time.perf_counter() - start

print("\nRepeated prediction:", prediction_repeat["text"].strip())
print(f"Repeated inference time: {repeat_seconds:.2f} seconds")
print(f"Repeated real-time factor: {repeat_seconds / duration:.2f}")

Audio duration: 6.34 seconds
Sample rate: 16000 Hz
First inference time: 23.26 seconds
First-run real-time factor: 3.67


[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Repeated prediction: വൈറസിനെ ജനങ്ങൾക്കിടിയിലേക്ക് ഇറക്കിവിട്ട് ഹെയ്ഡ് ഇമ്മ്യൂണിറ്റി നേടിയെടുക്കാം.
Repeated inference time: 13.82 seconds
Repeated real-time factor: 2.18


In [ ]:
print("Model generation configuration:")
print("max_length:", model.generation_config.max_length)
print("max_new_tokens:", model.generation_config.max_new_tokens)
print("num_beams:", model.generation_config.num_beams)
print("do_sample:", model.generation_config.do_sample)
print("forced_decoder_ids:", model.generation_config.forced_decoder_ids)

Model generation configuration:
max_length: 448
max_new_tokens: None
num_beams: None
do_sample: None
forced_decoder_ids: None


In [ ]:
# Remove the conflicting total-sequence length setting
model.generation_config.max_length = None

print("max_length:", model.generation_config.max_length)
print("max_new_tokens:", model.generation_config.max_new_tokens)
print("num_beams:", model.generation_config.num_beams)
print("do_sample:", model.generation_config.do_sample)

max_length: None
max_new_tokens: None
num_beams: None
do_sample: None


In [ ]:
import time
import torch

torch.cuda.synchronize()
start = time.perf_counter()

clean_prediction = asr(
    str(audio_path),
    generate_kwargs={
        "language": "malayalam",
        "task": "transcribe",
        "max_new_tokens": 444,
    },
)

torch.cuda.synchronize()
clean_elapsed = time.perf_counter() - start

print("Reference :", reference)
print("Prediction:", clean_prediction["text"].strip())
print(f"Inference time: {clean_elapsed:.2f} seconds")
print(f"Audio duration: {duration:.2f} seconds")
print(f"RTF: {clean_elapsed / duration:.2f}")

[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reference : വൈറസിനെ ജനങ്ങൾക്ക് ഇടയിലേക്ക് ഇറക്കി വിട്ട് ഹെർഡ് ഇമ്മ്യൂണിറ്റി നേടിയടുക്കാം
Prediction: വൈറസിനെ ജനങ്ങൾക്കിടിയിലേക്ക് ഇറക്കിവിട്ട് ഹെയ്ഡ് ഇമ്മ്യൂണിറ്റി നേടിയെടുക്കാം.
Inference time: 14.26 seconds
Audio duration: 6.34 seconds
RTF: 2.25


Evaluate 100 examples

In [ ]:
import time
import pandas as pd
import torch
import librosa
from pathlib import Path
from jiwer import wer, cer
from tqdm.auto import tqdm

results = []

# Warm up before collecting benchmark timings
_ = asr(
    str(evaluation_df.iloc[0]["audio_path"]),
    generate_kwargs={
        "language": "malayalam",
        "task": "transcribe",
        "max_new_tokens": 444,
    },
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

for _, row in tqdm(evaluation_df.iterrows(), total=len(evaluation_df)):
    audio_file = str(row["audio_path"])
    reference_text = str(row["sentence"]).strip()

    # Measure audio duration
    audio, sample_rate = librosa.load(
        audio_file, sr=16000, mono=True
    )
    audio_duration = len(audio) / sample_rate

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    output = asr(
        audio_file,
        generate_kwargs={
            "language": "malayalam",
            "task": "transcribe",
            "max_new_tokens": 444,
        },
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    predicted_text = output["text"].strip()

    results.append({
        "path": row["path"],
        "reference": reference_text,
        "prediction": predicted_text,
        "audio_duration_sec": audio_duration,
        "inference_sec": elapsed,
        "rtf": elapsed / audio_duration if audio_duration else None,
    })

results_df = pd.DataFrame(results)

# Aggregate error metrics across all examples
overall_wer = wer(
    results_df["reference"].tolist(),
    results_df["prediction"].tolist()
)
overall_cer = cer(
    results_df["reference"].tolist(),
    results_df["prediction"].tolist()
)

print(f"Evaluated examples: {len(results_df)}")
print(f"WER: {overall_wer:.4f} ({overall_wer * 100:.2f}%)")
print(f"CER: {overall_cer:.4f} ({overall_cer * 100:.2f}%)")
print(f"Mean inference time: {results_df['inference_sec'].mean():.2f} s")
print(f"Median inference time: {results_df['inference_sec'].median():.2f} s")
print(f"Mean RTF: {results_df['rtf'].mean():.3f}")
print(f"Overall RTF: {results_df['inference_sec'].sum() / results_df['audio_duration_sec'].sum():.3f}")

[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  0%|          | 0/100 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Evaluated examples: 100
WER: 0.5820 (58.20%)
CER: 0.1267 (12.67%)
Mean inference time: 2.52 s
Median inference time: 1.86 s
Mean RTF: 0.533
Overall RTF: 0.556


In [ ]:
from jiwer import wer, cer
from pathlib import Path

# Calculate errors for each recording individually
results_df["example_wer"] = results_df.apply(
    lambda r: wer(r["reference"], r["prediction"]), axis=1
)

results_df["example_cer"] = results_df.apply(
    lambda r: cer(r["reference"], r["prediction"]), axis=1
)

# Save the full results to Google Drive
RESULTS_DIR = Path(
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/experiment"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results_path = RESULTS_DIR / "baseline_results.csv"
results_df.to_csv(results_path, index=False, encoding="utf-8-sig")

print("Saved results:", results_path)
print("\nWorst 10 examples by word error rate:")

display(
    results_df.sort_values("example_wer", ascending=False)[
        [
            "path",
            "audio_duration_sec",
            "reference",
            "prediction",
            "example_wer",
            "example_cer",
            "inference_sec",
        ]
    ].head(10)
)

Saved results: /content/drive/MyDrive/vaaniResearch/malayalam_speech_ai/experiment/baseline_results.csv

Worst 10 examples by word error rate:


,path,audio_duration_sec,reference,prediction,example_wer,example_cer,inference_sec
54,common_voice_ml_37859541.mp3,2.268,ലണ്ടനിൽ തന്നെയല്ലേ,ലണ്ടിണി തന്നെ എല്ലേ,1.500000,0.277778,1.007264
49,common_voice_ml_37847037.mp3,4.176,യു.പി.എയ്ക്ക് കിട്ടിയത് ഇരുപത്തിമൂന്ന് ശതമാനം.,യു. പി. എയ്ക്ക് കിട്ടിയത് ഇരുപത്തി മൂന്നു ശ്തമ...,1.500000,0.108696,2.035136
70,common_voice_ml_37879010.mp3,3.600,അവിടെ കിട്ടുന്നതെന്തും അയച്ചുതരാനുള്ള,അവിടിക്കിട്ടുന്ന തെന്തും അയച്ചു തരാനുള്ള.,1.333333,0.162162,1.863082
53,common_voice_ml_37859511.mp3,3.636,നിനക്കെന്നോട് ദേഷ്യം വല്ലതുമുണ്ടോ?,നിനക്ക് ഇന്നോട് ദേക്ഷ്യം വലതുമുണ്ടോ,1.333333,0.235294,1.697302
66,common_voice_ml_37867286.mp3,4.680,ഹിന്ദുവര്ഗീയത എന്ന ആശയത്തിന് പൊതുസ്വീകാര്യതയുണ...,ഹിന്ദു വർഗ്ഗിയതെ എന്ന ആശയത്തിന് പൊതു സ്വീകാര്യ...,1.250000,0.226415,2.456218
25,common_voice_ml_37772166.mp3,3.384,ഞങ്ങൾക്ക് മറ്റൊരു ചോയ്സ് ഇല്ല,ഞങ്ങള്ക്ക് മറ്റൊരെ ചോയ്സില്ല,1.000000,0.206897,1.358132
60,common_voice_ml_37865174.mp3,3.168,ഞങ്ങളിതാ എത്തി,ഞങ്ങളിതായെത്തി,1.000000,0.142857,0.759540
46,common_voice_ml_37847009.mp3,5.364,ഫൈസാബാദിലെ ഉപതെരെഞ്ഞെടുപ്പാണ് അതില് എടുത്തുപറയ...,ഫൈസാബാദ് ലൂപതിനെ അടുപ്പാണ് അതിൽ എടുത്തുപറന്റെ ...,1.000000,0.306452,2.397029
61,common_voice_ml_37867200.mp3,3.996,അത് എപ്പോഴും ബുക്കുകള് തള്ളി ഇടാറുണ്ട്,അതെപ്പോഴും ബുക്കുകൾ തള്ളിയിടാറുണ്ട്,1.000000,0.184211,1.675190
73,common_voice_ml_37881497.mp3,3.060,നീയെന്താണ് നോക്കുന്നത്,നീ എന്താണ് നോക്കുന്നത്,1.000000,0.090909,1.168934


Experiment 1: Whisper decoding with beam search using 5 beams.

In [ ]:
import time
import torch
import pandas as pd
import librosa
from pathlib import Path
from tqdm.auto import tqdm

beam_results = []

# Warm up the model before measuring
_ = asr(
    str(evaluation_df.iloc[0]["audio_path"]),
    generate_kwargs={
        "language": "malayalam",
        "task": "transcribe",
        "num_beams": 5,
        "max_new_tokens": 444,
    },
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

for _, row in tqdm(evaluation_df.iterrows(), total=len(evaluation_df)):
    audio_file = str(row["audio_path"])
    reference_text = str(row["sentence"]).strip()

    audio, sample_rate = librosa.load(
        audio_file, sr=16000, mono=True
    )
    audio_duration = len(audio) / sample_rate

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    output = asr(
        audio_file,
        generate_kwargs={
            "language": "malayalam",
            "task": "transcribe",
            "num_beams": 5,
            "max_new_tokens": 444,
        },
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    beam_results.append({
        "path": row["path"],
        "reference": reference_text,
        "prediction": output["text"].strip(),
        "audio_duration_sec": audio_duration,
        "inference_sec": elapsed,
        "rtf": elapsed / audio_duration if audio_duration else None,
    })

beam_df = pd.DataFrame(beam_results)

experiment_dir = Path(
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/experiment"
)
experiment_dir.mkdir(parents=True, exist_ok=True)

beam_path = experiment_dir / "beam5_results.csv"
beam_df.to_csv(beam_path, index=False, encoding="utf-8-sig")

print("Saved:", beam_path)
print("Examples:", len(beam_df))

[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  0%|          | 0/100 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Saved: /content/drive/MyDrive/vaaniResearch/malayalam_speech_ai/experiment/beam5_results.csv
Examples: 100


In [ ]:
from jiwer import wer, cer

baseline_wer = wer(
    results_df["reference"].tolist(),
    results_df["prediction"].tolist()
)
baseline_cer = cer(
    results_df["reference"].tolist(),
    results_df["prediction"].tolist()
)

beam_wer = wer(
    beam_df["reference"].tolist(),
    beam_df["prediction"].tolist()
)
beam_cer = cer(
    beam_df["reference"].tolist(),
    beam_df["prediction"].tolist()
)

comparison = pd.DataFrame([
    {
        "configuration": "Baseline",
        "WER_percent": baseline_wer * 100,
        "CER_percent": baseline_cer * 100,
        "mean_inference_sec": results_df["inference_sec"].mean(),
        "overall_RTF": (
            results_df["inference_sec"].sum()
            / results_df["audio_duration_sec"].sum()
        ),
    },
    {
        "configuration": "Beam search (5 beams)",
        "WER_percent": beam_wer * 100,
        "CER_percent": beam_cer * 100,
        "mean_inference_sec": beam_df["inference_sec"].mean(),
        "overall_RTF": (
            beam_df["inference_sec"].sum()
            / beam_df["audio_duration_sec"].sum()
        ),
    },
])

display(comparison.round(3))

,configuration,WER_percent,CER_percent,mean_inference_sec,overall_RTF
0,Baseline,58.2,12.672,2.521,0.556
1,Beam search (5 beams),58.2,12.672,2.138,0.472


In [ ]:
# Check whether the two configurations produced different transcripts
comparison_df = results_df[
    ["path", "reference", "prediction"]
].merge(
    beam_df[["path", "prediction"]],
    on="path",
    suffixes=("_baseline", "_beam5"),
    validate="one_to_one"
)

different = comparison_df[
    comparison_df["prediction_baseline"]
    != comparison_df["prediction_beam5"]
]

print("Examples compared:", len(comparison_df))
print("Transcripts that changed:", len(different))

display(
    different[
        ["reference", "prediction_baseline", "prediction_beam5"]
    ].head(10)
)

print("\nModel generation configuration:")
print(model.generation_config)

Examples compared: 100
Transcripts that changed: 0


,reference,prediction_baseline,prediction_beam5



Model generation configuration:
GenerationConfig {
  "alignment_heads": [
    [
      5,
      3
    ],
    [
      5,
      9
    ],
    [
      8,
      0
    ],
    [
      8,
      4
    ],
    [
      8,
      7
    ],
    [
      8,
      8
    ],
    [
      9,
      0
    ],
    [
      9,
      7
    ],
    [
      9,
      9
    ],
    [
      10,
      5
    ]
  ],
  "begin_suppress_tokens": [
    220,
    50257
  ],
  "bos_token_id": 50257,
  "decoder_start_token_id": 50258,
  "eos_token_id": 50257,
  "forced_decoder_ids": null,
  "is_multilingual": true,
  "lang_to_id": {
    "<|af|>": 50327,
    "<|am|>": 50334,
    "<|ar|>": 50272,
    "<|as|>": 50350,
    "<|az|>": 50304,
    "<|ba|>": 50355,
    "<|be|>": 50330,
    "<|bg|>": 50292,
    "<|bn|>": 50302,
    "<|bo|>": 50347,
    "<|br|>": 50309,
    "<|bs|>": 50315,
    "<|ca|>": 50270,
    "<|cs|>": 50283,
    "<|cy|>": 50297,
    "<|da|>": 50285,
    "<|de|>": 50261,
    "<|el|>": 50281,
    "<|en|>": 50259,
    "<|e

In [ ]:
print("Pipeline model:", type(asr.model).__name__)
print("Pipeline generation config:")
print(asr.model.generation_config)

print("\nPipeline task:", asr.task)
print("Pipeline device:", asr.device)

Pipeline model: WhisperForConditionalGeneration
Pipeline generation config:
GenerationConfig {
  "alignment_heads": [
    [
      5,
      3
    ],
    [
      5,
      9
    ],
    [
      8,
      0
    ],
    [
      8,
      4
    ],
    [
      8,
      7
    ],
    [
      8,
      8
    ],
    [
      9,
      0
    ],
    [
      9,
      7
    ],
    [
      9,
      9
    ],
    [
      10,
      5
    ]
  ],
  "begin_suppress_tokens": [
    220,
    50257
  ],
  "bos_token_id": 50257,
  "decoder_start_token_id": 50258,
  "eos_token_id": 50257,
  "forced_decoder_ids": null,
  "is_multilingual": true,
  "lang_to_id": {
    "<|af|>": 50327,
    "<|am|>": 50334,
    "<|ar|>": 50272,
    "<|as|>": 50350,
    "<|az|>": 50304,
    "<|ba|>": 50355,
    "<|be|>": 50330,
    "<|bg|>": 50292,
    "<|bn|>": 50302,
    "<|bo|>": 50347,
    "<|br|>": 50309,
    "<|bs|>": 50315,
    "<|ca|>": 50270,
    "<|cs|>": 50283,
    "<|cy|>": 50297,
    "<|da|>": 50285,
    "<|de|>": 50261,
    "<|

## Experiment 2 — Compare Malayalam Whisper Checkpoints

### Objective
Test whether the Malayalam-specific Whisper v2 checkpoint improves transcription accuracy over our baseline model.

### Hypothesis
The v2 checkpoint may reduce transcription errors, but this is not guaranteed. We will measure the outcome rather than assume it is better.

### Controlled experiment
- **Baseline:** `sajilck/whisper-small-malayalam`
- **Candidate:** `sajilck/whisper-small-malayalam-v2`
- **Evaluation set:** the same 100 recordings in `evaluation_manifest.csv`
- **Metrics:** Word Error Rate (WER), Character Error Rate (CER), inference time, and Real-Time Factor (RTF)
- **Controls:** same audio files, reference transcripts, evaluation procedure, and GPU environment

### Decision rule
Keep the candidate only if the measured results justify it. Report negative results honestly.


In [ ]:
from huggingface_hub import model_info

MODEL_ID_V2 = "sajilck/whisper-small-malayalam-v2"

try:
    info = model_info(MODEL_ID_V2)
    print("Model:", info.id)
    print("Latest revision:", info.sha)
    print("Model tags:", info.tags)
except Exception as e:
    print("Could not retrieve model information:")
    print(type(e).__name__, str(e))

Model: sajilck/whisper-small-malayalam-v2
Latest revision: 77356a144f325ad505477d392ed8e5eb46af194d
Model tags: ['safetensors', 'whisper', 'automatic-speech-recognition', 'malayalam', 'low-resource-asr', 'indic-languages', 'ggml', 'whisper-cpp', 'ml', 'dataset:sajilck/malayalam-asr-corpus', 'base_model:openai/whisper-small', 'base_model:finetune:openai/whisper-small', 'license:apache-2.0', 'model-index', 'region:us']


In [ ]:
import os
import torch
import pandas as pd
from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    pipeline,
)

# Exact model revision from your inspection
MODEL_ID_V2 = "sajilck/whisper-small-malayalam-v2"
MODEL_REVISION_V2 = "77356a144f325ad505477d392ed8e5eb46af194d"

# Locations already used in our experiment
DATA_DIR = (
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/ml"
)
MANIFEST_PATH = (
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/experiment/evaluation_manifest.csv"
)

# Load the same fixed 100-example manifest
manifest_df = pd.read_csv(MANIFEST_PATH)
sample = manifest_df.iloc[0]

audio_path = os.path.join(DATA_DIR, "clips", sample["path"])
reference = sample["sentence"] if "sentence" in sample.index else None

print("Audio exists:", os.path.exists(audio_path))
print("Audio file:", sample["path"])

# Load the processor and v2 model
processor_v2 = AutoProcessor.from_pretrained(
    MODEL_ID_V2,
    revision=MODEL_REVISION_V2,
)

model_v2 = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID_V2,
    revision=MODEL_REVISION_V2,
    torch_dtype=torch.float16,
).to("cuda")

asr_v2 = pipeline(
    "automatic-speech-recognition",
    model=model_v2,
    tokenizer=processor_v2.tokenizer,
    feature_extractor=processor_v2.feature_extractor,
    device=0,
    torch_dtype=torch.float16,
    generate_kwargs={
        "language": "malayalam",
        "task": "transcribe",
        "max_new_tokens": 444,
    },
)

# Test only one recording first
result_v2 = asr_v2(audio_path)

print("\nReference:", reference)
print("V2 prediction:", result_v2["text"])

Audio exists: True
Audio file: common_voice_ml_30346653.mp3


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.79k [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Reference: വൈറസിനെ ജനങ്ങൾക്ക് ഇടയിലേക്ക് ഇറക്കി വിട്ട് ഹെർഡ് ഇമ്മ്യൂണിറ്റി നേടിയടുക്കാം
V2 prediction: വൈറസിനെ ജനങ്ങൾക്കിടയിലേക്ക് ഇറക്കി വിട്ട് ഹെയ്ഡ് ഇമ്മ്യൂണിറ്റി നേടിയെടുക്കാം.


In [ ]:
import os
import time
import pandas as pd
import soundfile as sf
import jiwer

RESULTS_DIR = (
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/experiment"
)
os.makedirs(RESULTS_DIR, exist_ok=True)

manifest_df = pd.read_csv(MANIFEST_PATH)

# Identify the reference-transcript column.
print("Manifest columns:", manifest_df.columns.tolist())

Manifest columns: ['path', 'audio_path', 'sentence', 'client_id', 'sentence_id', 'up_votes', 'down_votes']


In [ ]:
import os
import time
import pandas as pd
import jiwer

RESULTS_DIR = (
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/experiment"
)
os.makedirs(RESULTS_DIR, exist_ok=True)

manifest_df = pd.read_csv(MANIFEST_PATH)

# Warm up the model before measuring latency.
warmup_path = os.path.join(DATA_DIR, "clips", manifest_df.iloc[0]["path"])
_ = asr_v2(warmup_path)

rows = []

for i, row in manifest_df.iterrows():
    audio_path = os.path.join(DATA_DIR, "clips", row["path"])

    if not os.path.exists(audio_path):
        print(f"Skipping missing file: {row['path']}")
        continue

    start = time.perf_counter()
    result = asr_v2(audio_path)
    elapsed = time.perf_counter() - start

    # Read duration from the audio file.
    audio_info = sf.info(audio_path)
    duration = audio_info.duration

    rows.append({
        "path": row["path"],
        "sentence_id": row["sentence_id"],
        "reference": row["sentence"],
        "prediction": result["text"],
        "inference_seconds": elapsed,
        "audio_seconds": duration,
        "rtf": elapsed / duration if duration > 0 else None,
    })

    if (len(rows) % 10) == 0:
        print(f"Processed {len(rows)}/{len(manifest_df)}")

v2_df = pd.DataFrame(rows)

# Calculate corpus-level WER and CER.
references = v2_df["reference"].fillna("").tolist()
predictions = v2_df["prediction"].fillna("").tolist()

v2_wer = jiwer.wer(references, predictions)
v2_cer = jiwer.cer(references, predictions)

total_audio_seconds = v2_df["audio_seconds"].sum()
total_inference_seconds = v2_df["inference_seconds"].sum()

print("\n--- V2 RESULTS ---")
print("Evaluated examples:", len(v2_df))
print(f"WER: {v2_wer:.4f} ({v2_wer * 100:.2f}%)")
print(f"CER: {v2_cer:.4f} ({v2_cer * 100:.2f}%)")
print(f"Mean inference time: {v2_df['inference_seconds'].mean():.3f} sec")
print(f"Median inference time: {v2_df['inference_seconds'].median():.3f} sec")
print(f"Mean RTF: {v2_df['rtf'].mean():.3f}")
print(f"Overall RTF: {total_inference_seconds / total_audio_seconds:.3f}")

output_path = os.path.join(RESULTS_DIR, "v2_results.csv")
v2_df.to_csv(output_path, index=False)
print("\nSaved results to:", output_path)

[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 10/100


[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 20/100


[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

...

Processed 80/100


[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 90/100


[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Processed 100/100

--- V2 RESULTS ---
Evaluated examples: 100
WER: 0.2340 (23.40%)
CER: 0.0375 (3.75%)
Mean inference time: 2.141 sec
Median inference time: 1.663 sec
Mean RTF: 0.444
Overall RTF: 0.471

Saved results to: /content/drive/MyDrive/vaaniResearch/malayalam_speech_ai/experiment/v2_results.csv


In [ ]:
import os
import pandas as pd
import jiwer

experiment_dir = (
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/experiment"
)

baseline_df = pd.read_csv(
    os.path.join(experiment_dir, "baseline_results.csv")
)
v2_df = pd.read_csv(
    os.path.join(experiment_dir, "v2_results.csv")
)

# Keep only the columns needed for the comparison.
comparison = baseline_df[
    ["path", "reference", "prediction"]
].merge(
    v2_df[["path", "reference", "prediction"]],
    on="path",
    how="inner",
    suffixes=("_baseline", "_v2"),
    validate="one_to_one",
)

# Verify that the references really are identical.
assert len(comparison) == 100, (
    f"Expected 100 matching recordings; found {len(comparison)}"
)
assert (
    comparison["reference_baseline"].fillna("").tolist()
    == comparison["reference_v2"].fillna("").tolist()
), "Reference transcripts differ between runs!"

comparison["baseline_word_error"] = comparison.apply(
    lambda r: jiwer.wer(
        str(r["reference_baseline"]), str(r["prediction_baseline"])
    ),
    axis=1,
)

comparison["v2_word_error"] = comparison.apply(
    lambda r: jiwer.wer(
        str(r["reference_v2"]), str(r["prediction_v2"])
    ),
    axis=1,
)

comparison["wer_improvement"] = (
    comparison["baseline_word_error"] - comparison["v2_word_error"]
)

print("Matching recordings:", len(comparison))
print(
    "Transcripts changed:",
    (
        comparison["prediction_baseline"]
        != comparison["prediction_v2"]
    ).sum(),
)
print(
    "Examples with lower per-example WER:",
    (comparison["wer_improvement"] > 0).sum(),
)
print(
    "Examples with higher per-example WER:",
    (comparison["wer_improvement"] < 0).sum(),
)

# Inspect five examples with the largest WER improvement.
print("\nExamples with the largest improvement:")
display(
    comparison.sort_values("wer_improvement", ascending=False)[
        [
            "reference_baseline",
            "prediction_baseline",
            "prediction_v2",
            "wer_improvement",
        ]
    ].head(5)
)

Matching recordings: 100
Transcripts changed: 81
Examples with lower per-example WER: 66
Examples with higher per-example WER: 5

Examples with the largest improvement:


,reference_baseline,prediction_baseline,prediction_v2,wer_improvement
54,ലണ്ടനിൽ തന്നെയല്ലേ,ലണ്ടിണി തന്നെ എല്ലേ,ലണ്ടനിൽ തന്നെയല്ലേ,1.500000
53,നിനക്കെന്നോട് ദേഷ്യം വല്ലതുമുണ്ടോ?,നിനക്ക് ഇന്നോട് ദേക്ഷ്യം വലതുമുണ്ടോ,നിനക്കെന്നോട് ദേഷ്യം വല്ലതുമുണ്ടോ?,1.333333
70,അവിടെ കിട്ടുന്നതെന്തും അയച്ചുതരാനുള്ള,അവിടിക്കിട്ടുന്ന തെന്തും അയച്ചു തരാനുള്ള.,അവിടെ കിട്ടുന്നതെന്തും അയച്ചുതരാനുള്ള,1.333333
49,യു.പി.എയ്ക്ക് കിട്ടിയത് ഇരുപത്തിമൂന്ന് ശതമാനം.,യു. പി. എയ്ക്ക് കിട്ടിയത് ഇരുപത്തി മൂന്നു ശ്തമ...,യു.ബി.എയ്ക്ക് കിട്ടിയത് ഇരുപത്തിമൂന്ന് ശതമാനം.,1.250000
41,നീയും മാറരുത്,നീയം മാറച്ചത്,നീയും മാറരുത്,1.000000


### Baseline vs. Malayalam Whisper v2

We evaluated both checkpoints on the same fixed set of 100 Malayalam Common Voice test recordings. The baseline achieved 58.20% WER and 12.67% CER, whereas the v2 checkpoint achieved 23.40% WER and 3.75% CER. This corresponds to a relative WER reduction of approximately 59.8% and a relative CER reduction of approximately 70.4%.

At the individual-recording level, 66 examples had lower WER with v2, five had higher WER, and 29 had unchanged WER. Transcriptions changed for 81 recordings. Qualitative inspection showed improvements in spelling and word segmentation, although errors involving abbreviations remained.

Mean inference time was 2.180 seconds for the baseline and 2.208 seconds for v2. Overall RTF was 0.481 and 0.486, respectively. These small timing differences do not establish a meaningful latency difference.

The results favor v2 for transcription accuracy on this sample. However, the evaluation is limited to 100 recordings from a scripted-speech dataset, and results may not generalize to spontaneous conversation, different recording conditions, or all Malayalam speaker groups.


PART 2

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_ID = "Qwen/Qwen2.5-3B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_ID)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Model loaded:", LLM_ID)
print("Device:", llm_model.device)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-3B-Instruct
Device: cuda:0


In [ ]:
def generate_answer(question, max_new_tokens=100):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. "
                "Answer clearly and concisely in Malayalam. "
                "If you do not know an answer, say so rather "
                "than inventing facts."
            ),
        },
        {"role": "user", "content": question},
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
    ).to(llm_model.device)

    with torch.inference_mode():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    # Decode only newly generated tokens, not the input prompt.
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return llm_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()


question = "കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?"
answer = generate_answer(question)

print("Question:", question)
print("Answer:", answer)

Question: കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?
Answer: കേരളത്തിന്റെ തലസ്ഥാനം അലപ്പുറത്തു നടക്കുന്ന മഹാകലയം കേരളത്തിലെ തലസ്ഥാന കലാമൂർ


In [ ]:
def generate_answer_v2(question, max_new_tokens=80):
    messages = [
        {
            "role": "system",
            "content": (
                "Answer the user's question with one short factual sentence. "
                "Use simple, grammatically correct Malayalam. "
                "Do not add unrelated information. "
                "If you are unsure, say you do not know."
            ),
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
    ).to(llm_model.device)

    with torch.inference_mode():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]

    return llm_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()


print(generate_answer_v2("കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?"))

കേരളത്തിന്റെ തലസ്ഥാനം കടലുകൂടിയായിരുന്നു.


In [ ]:
print(generate_answer_v2("What is the capital of Kerala?"))

Thiruvananthapuram തിരുവനന്തപുരം 是 Kerala के कैपिटल.


In [ ]:
!nvidia-smi

Sat Oct 24 15:43:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P0             29W /   70W |    6103MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import gc
import torch

print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved:", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

# Clear unused cached memory
gc.collect()
torch.cuda.empty_cache()

print("After cleanup:")
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved:", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

Allocated: 5.76 GB
Reserved: 5.83 GB
After cleanup:
Allocated: 5.76 GB
Reserved: 5.82 GB


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Loaded:", MODEL_ID)
print("Device:", llm_model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0


In [ ]:
questions = [
    {
        "question": "കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?",
        "expected": "തിരുവനന്തപുരം"
    },
    {
        "question": "ഇന്ത്യയുടെ തലസ്ഥാനം ഏതാണ്?",
        "expected": "ന്യൂഡൽഹി"
    },
    {
        "question": "ഒരു ആഴ്ചയിൽ എത്ര ദിവസങ്ങളുണ്ട്?",
        "expected": "7"
    },
    {
        "question": "കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ ഏതാണ്?",
        "expected": "മലയാളം"
    },
    {
        "question": "സൂര്യൻ ഉദിക്കുന്നത് ഏത് ദിശയിലാണ്?",
        "expected": "കിഴക്ക്"
    }
]

def answer_malayalam(question):
    messages = [
        {
            "role": "system",
            "content": (
                "നിങ്ങൾ മലയാളത്തിൽ മാത്രം മറുപടി നൽകുന്ന ഒരു സഹായിയാണ്. "
                "ചോദിച്ചതിന് കൃത്യവും ഹ്രസ്വവുമായ മറുപടി നൽകുക. "
                "അനാവശ്യ വിവരങ്ങൾ ചേർക്കരുത്."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return llm_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

for i, item in enumerate(questions, start=1):
    answer = answer_malayalam(item["question"])
    print(f"{i}. Question: {item['question']}")
    print(f"   Expected: {item['expected']}")
    print(f"   Model:    {answer}")
    print()

1. Question: കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?
   Expected: തിരുവനന്തപുരം
   Model:    കേരളത്തിന്റെ തലസ്ഥാനം പഞ്ചാബദ്ധതയും ഉണ്ടായിര

2. Question: ഇന്ത്യയുടെ തലസ്ഥാനം ഏതാണ്?
   Expected: ന്യൂഡൽഹി
   Model:    ഇന്ത്യയുടെ തലസ്ഥാനം പഞ്ചാബദ്ധതയും ഉണ്ടായിരു

3. Question: ഒരു ആഴ്ചയിൽ എത്ര ദിവസങ്ങളുണ്ട്?
   Expected: 7
   Model:    ആഴക്കം 31 ദിവസം നൽകുന്നു.

4. Question: കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ ഏതാണ്?
   Expected: മലയാളം
   Model:    കേരളത്തിന്റെ പഠിക്കപ്പെട്ട ഉദ്യോഗിക ഭാഷ എന്നിവ

5. Question: സൂര്യൻ ഉദിക്കുന്നത് ഏത് ദിശയിലാണ്?
   Expected: കിഴക്ക്
   Model:    സൂര്യൻ എന്ന പദം ആദ്യം ജനിച്ച സൂര്യന്റെ പ്രധാന �



In [ ]:
 import torch

messages = [
    {
        "role": "system",
        "content": (
            "Answer in Malayalam using only the supplied context. "
            "Give one short answer. Do not invent information."
        )
    },
    {
        "role": "user",
        "content": (
            "Context: കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരമാണ്.\n"
            "Question: കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?"
        )
    }
]

prompt = llm_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = llm_tokenizer(
    prompt,
    return_tensors="pt"
).to(llm_model.device)

with torch.no_grad():
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False
    )

answer = llm_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Answer:", answer)


Answer: കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരമാണ്.


In [ ]:
import pandas as pd
from pathlib import Path

qa_rows = [
    ("കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?", "തിരുവനന്തപുരം",
     "കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരം ആണ്."),
    ("ഇന്ത്യയുടെ തലസ്ഥാനം ഏതാണ്?", "ന്യൂഡൽഹി",
     "ഇന്ത്യയുടെ തലസ്ഥാനം ന്യൂഡൽഹി ആണ്."),
    ("കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ ഏതാണ്?", "മലയാളം",
     "കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ മലയാളം ആണ്."),
    ("ഒരു ആഴ്ചയിൽ എത്ര ദിവസങ്ങളുണ്ട്?", "7",
     "ഒരു ആഴ്ചയിൽ ഏഴ് ദിവസങ്ങളുണ്ട്."),
    ("സൂര്യൻ ഉദിക്കുന്നത് ഏത് ദിശയിലാണ്?", "കിഴക്ക്",
     "സൂര്യൻ കിഴക്ക് ദിശയിൽ ഉദിക്കുന്നു."),
    ("ഇന്ത്യയുടെ നാണയം ഏതാണ്?", "രൂപ",
     "ഇന്ത്യയുടെ ഔദ്യോഗിക നാണയം ഇന്ത്യൻ രൂപയാണ്."),
    ("ഒരു വർഷത്തിൽ സാധാരണ എത്ര മാസങ്ങളുണ്ട്?", "12",
     "ഒരു വർഷത്തിൽ പന്ത്രണ്ട് മാസങ്ങളുണ്ട്."),
    ("കേരളം ഇന്ത്യയുടെ ഏത് ഭാഗത്താണ്?", "തെക്കുപടിഞ്ഞാറ്",
     "കേരളം ഇന്ത്യയുടെ തെക്കുപടിഞ്ഞാറൻ തീരത്താണ്."),
    ("വെള്ളത്തിന്റെ രാസസൂത്രം എന്താണ്?", "H2O",
     "വെള്ളത്തിന്റെ രാസസൂത്രം H2O ആണ്."),
    ("സസ്യങ്ങൾ സൂര്യപ്രകാശം ഉപയോഗിച്ച് ആഹാരം നിർമ്മിക്കുന്ന പ്രക്രിയ ഏതാണ്?",
     "പ്രകാശസംശ്ലേഷണം",
     "സസ്യങ്ങൾ പ്രകാശസംശ്ലേഷണത്തിലൂടെ ആഹാരം നിർമ്മിക്കുന്നു.")
]

qa_df = pd.DataFrame(
    qa_rows,
    columns=["question", "expected_answer", "verified_context"]
)

out_dir = Path(
    "/content/drive/MyDrive/vaaniResearch/malayalam_speech_ai/experiment"
)
out_dir.mkdir(parents=True, exist_ok=True)

qa_df.to_csv(out_dir / "qa_dataset_seed.csv", index=False)
print(f"Saved {len(qa_df)} initial questions to {out_dir}")
display(qa_df)

Saved 10 initial questions to /content/drive/MyDrive/vaaniResearch/malayalam_speech_ai/experiment


,question,expected_answer,verified_context
0,കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?,തിരുവനന്തപുരം,കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരം ആണ്.
1,ഇന്ത്യയുടെ തലസ്ഥാനം ഏതാണ്?,ന്യൂഡൽഹി,ഇന്ത്യയുടെ തലസ്ഥാനം ന്യൂഡൽഹി ആണ്.
2,കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ ഏതാണ്?,മലയാളം,കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ മലയാളം ആണ്.
3,ഒരു ആഴ്ചയിൽ എത്ര ദിവസങ്ങളുണ്ട്?,7,ഒരു ആഴ്ചയിൽ ഏഴ് ദിവസങ്ങളുണ്ട്.
4,സൂര്യൻ ഉദിക്കുന്നത് ഏത് ദിശയിലാണ്?,കിഴക്ക്,സൂര്യൻ കിഴക്ക് ദിശയിൽ ഉദിക്കുന്നു.
5,ഇന്ത്യയുടെ നാണയം ഏതാണ്?,രൂപ,ഇന്ത്യയുടെ ഔദ്യോഗിക നാണയം ഇന്ത്യൻ രൂപയാണ്.
6,ഒരു വർഷത്തിൽ സാധാരണ എത്ര മാസങ്ങളുണ്ട്?,12,ഒരു വർഷത്തിൽ പന്ത്രണ്ട് മാസങ്ങളുണ്ട്.
7,കേരളം ഇന്ത്യയുടെ ഏത് ഭാഗത്താണ്?,തെക്കുപടിഞ്ഞാറ്,കേരളം ഇന്ത്യയുടെ തെക്കുപടിഞ്ഞാറൻ തീരത്താണ്.
8,വെള്ളത്തിന്റെ രാസസൂത്രം എന്താണ്?,H2O,വെള്ളത്തിന്റെ രാസസൂത്രം H2O ആണ്.
9,സസ്യങ്ങൾ സൂര്യപ്രകാശം ഉപയോഗിച്ച് ആഹാരം നിർമ്മി...,പ്രകാശസംശ്ലേഷണം,സസ്യങ്ങൾ പ്രകാശസംശ്ലേഷണത്തിലൂടെ ആഹാരം നിർമ്മിക...


In [ ]:
import pandas as pd
import torch
import time
from pathlib import Path

QA_PATH = (
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/experiment/qa_dataset_seed.csv"
)
OUT_DIR = Path(
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/experiment"
)

qa_df = pd.read_csv(QA_PATH)

def generate_answer(question, context=None):
    if context:
        system_prompt = (
            "Answer in Malayalam using only the supplied context. "
            "Give one short answer. If the context does not contain "
            "the answer, say you do not know."
        )
        user_prompt = f"Context: {context}\nQuestion: {question}"
    else:
        system_prompt = (
            "Answer the question briefly in Malayalam. "
            "Do not add unrelated information."
        )
        user_prompt = question

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = llm_tokenizer(
        prompt, return_tensors="pt"
    ).to(llm_model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    answer = llm_tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return answer, elapsed

records = []

for i, row in qa_df.iterrows():
    baseline, baseline_s = generate_answer(row["question"])
    assisted, assisted_s = generate_answer(
        row["question"], row["verified_context"]
    )

    records.append({
        "question": row["question"],
        "expected_answer": row["expected_answer"],
        "verified_context": row["verified_context"],
        "baseline_answer": baseline,
        "baseline_latency_s": baseline_s,
        "context_answer": assisted,
        "context_latency_s": assisted_s
    })

results_df = pd.DataFrame(records)
results_path = OUT_DIR / "qa_seed_comparison.csv"
results_df.to_csv(results_path, index=False)

print("Saved:", results_path)
display(results_df)

Saved: /content/drive/MyDrive/vaaniResearch/malayalam_speech_ai/experiment/qa_seed_comparison.csv


,question,expected_answer,verified_context,baseline_answer,baseline_latency_s,context_answer,context_latency_s
0,കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?,തിരുവനന്തപുരം,കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരം ആണ്.,കേരളത്തിന്റെ തലസ്ഥാനം കൊടുപ്പായി 13 മീറ്റർ വൃത...,4.627508,കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരം ആണ്.,5.559395
1,ഇന്ത്യയുടെ തലസ്ഥാനം ഏതാണ്?,ന്യൂഡൽഹി,ഇന്ത്യയുടെ തലസ്ഥാനം ന്യൂഡൽഹി ആണ്.,ഇന്ത്യയുടെ തലസ്ഥാനം പഞ്ചകാലികളിൽ നിർമ്മിച്ച കോ...,3.668483,ഇന്ത്യയുടെ തലസ്ഥാനം ന്യൂഡൽഹി ആണ്.,1.815542
2,കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ ഏതാണ്?,മലയാളം,കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ മലയാളം ആണ്.,കേരളത്തിന്റെ പല സമയങ്ങൾ കൊടുത്ത ഉദ്യോഗിക ഭാഷ എ...,3.221923,കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ എന്നാൽ മലയാളം ആണ്.,2.455735
3,ഒരു ആഴ്ചയിൽ എത്ര ദിവസങ്ങളുണ്ട്?,7,ഒരു ആഴ്ചയിൽ ഏഴ് ദിവസങ്ങളുണ്ട്.,"ഏതാം, തടിച്ചാലോകനം നിർമ്മിക്കപ്പെടുന്നതിനാൽ ഇത...",3.146108,ഏഴ്ചയിൽ ഏഴ് ദിവസങ്ങളുണ്ട്.,1.287717
4,സൂര്യൻ ഉദിക്കുന്നത് ഏത് ദിശയിലാണ്?,കിഴക്ക്,സൂര്യൻ കിഴക്ക് ദിശയിൽ ഉദിക്കുന്നു.,സൂര്യൻ മുഴുവന്റെ സമൃദ്ധി എന്നിവയാണ് പറയുന്നത്.,2.420999,സൂര്യൻ ദിശയിൽ ഉദിക്കുന്നു.,1.394358
5,ഇന്ത്യയുടെ നാണയം ഏതാണ്?,രൂപ,ഇന്ത്യയുടെ ഔദ്യോഗിക നാണയം ഇന്ത്യൻ രൂപയാണ്.,ഇന്ത്യയുടെ നാണയം പഞ്ചദാബദ്ധമായിരുന്നു.,2.133448,ഇന്ത്യൻ രൂപയാണ്.,0.875901
6,ഒരു വർഷത്തിൽ സാധാരണ എത്ര മാസങ്ങളുണ്ട്?,12,ഒരു വർഷത്തിൽ പന്ത്രണ്ട് മാസങ്ങളുണ്ട്.,4 മാസങ്ങൾ.,0.532078,ഏതാം മാസങ്ങൾക്കെതിരെ 12 മാസങ്ങൾക്കേതിരെ.,2.071336
7,കേരളം ഇന്ത്യയുടെ ഏത് ഭാഗത്താണ്?,തെക്കുപടിഞ്ഞാറ്,കേരളം ഇന്ത്യയുടെ തെക്കുപടിഞ്ഞാറൻ തീരത്താണ്.,കേരളം ഈസ്റ്റിയയുടെ കൊല്ലാശൈലി ഭാഗത്താണ്.,2.243325,കേരളം ഇന്ത്യയുടെ തെക്കുപടിഞ്ഞാറൻ തീരത്താണ്.,2.312783
8,വെള്ളത്തിന്റെ രാസസൂത്രം എന്താണ്?,H2O,വെള്ളത്തിന്റെ രാസസൂത്രം H2O ആണ്.,വെള്ളത്തിന്റെ രാസസൂത്രം പുരുഷായിക സൂത്രമായിരിക...,3.222373,H2O ആണ്.,0.434489
9,സസ്യങ്ങൾ സൂര്യപ്രകാശം ഉപയോഗിച്ച് ആഹാരം നിർമ്മി...,പ്രകാശസംശ്ലേഷണം,സസ്യങ്ങൾ പ്രകാശസംശ്ലേഷണത്തിലൂടെ ആഹാരം നിർമ്മിക...,വിദ്യാർത്ഥിക അനേകം സസ്യങ്ങളുടെ സൂര്യപ്രകാശം ഉപ...,3.114907,സസ്യങ്ങൾ സൂര്യപ്രകാശം ഉപയോഗിച്ച് ആഹാരം നിർമ്മി...,3.099133


In [ ]:
import pandas as pd
import unicodedata
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/vaaniResearch/"
    "malayalam_speech_ai/experiment/qa_seed_comparison.csv"
)

df = pd.read_csv(path)

def normalize(text):
    text = unicodedata.normalize("NFC", str(text)).casefold()
    return " ".join(text.strip().split())

def contains_expected(answer, expected):
    answer = normalize(answer)
    expected = normalize(expected)
    return expected in answer

df["baseline_match"] = df.apply(
    lambda r: contains_expected(
        r["baseline_answer"], r["expected_answer"]
    ),
    axis=1
)

df["context_match"] = df.apply(
    lambda r: contains_expected(
        r["context_answer"], r["expected_answer"]
    ),
    axis=1
)

print("Baseline exact-answer substring rate:",
      round(df["baseline_match"].mean() * 100, 1), "%")
print("Context-assisted exact-answer substring rate:",
      round(df["context_match"].mean() * 100, 1), "%")
print("Mean baseline latency:",
      round(df["baseline_latency_s"].mean(), 3), "seconds")
print("Mean context-assisted latency:",
      round(df["context_latency_s"].mean(), 3), "seconds")

display(df[[
    "question", "expected_answer",
    "baseline_answer", "baseline_match",
    "context_answer", "context_match"
]])

Baseline exact-answer substring rate: 0.0 %
Context-assisted exact-answer substring rate: 60.0 %
Mean baseline latency: 2.833 seconds
Mean context-assisted latency: 2.131 seconds


,question,expected_answer,baseline_answer,baseline_match,context_answer,context_match
0,കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?,തിരുവനന്തപുരം,കേരളത്തിന്റെ തലസ്ഥാനം കൊടുപ്പായി 13 മീറ്റർ വൃത...,False,കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരം ആണ്.,True
1,ഇന്ത്യയുടെ തലസ്ഥാനം ഏതാണ്?,ന്യൂഡൽഹി,ഇന്ത്യയുടെ തലസ്ഥാനം പഞ്ചകാലികളിൽ നിർമ്മിച്ച കോ...,False,ഇന്ത്യയുടെ തലസ്ഥാനം ന്യൂഡൽഹി ആണ്.,True
2,കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ ഏതാണ്?,മലയാളം,കേരളത്തിന്റെ പല സമയങ്ങൾ കൊടുത്ത ഉദ്യോഗിക ഭാഷ എ...,False,കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ എന്നാൽ മലയാളം ആണ്.,True
3,ഒരു ആഴ്ചയിൽ എത്ര ദിവസങ്ങളുണ്ട്?,7,"ഏതാം, തടിച്ചാലോകനം നിർമ്മിക്കപ്പെടുന്നതിനാൽ ഇത...",False,ഏഴ്ചയിൽ ഏഴ് ദിവസങ്ങളുണ്ട്.,False
4,സൂര്യൻ ഉദിക്കുന്നത് ഏത് ദിശയിലാണ്?,കിഴക്ക്,സൂര്യൻ മുഴുവന്റെ സമൃദ്ധി എന്നിവയാണ് പറയുന്നത്.,False,സൂര്യൻ ദിശയിൽ ഉദിക്കുന്നു.,False
5,ഇന്ത്യയുടെ നാണയം ഏതാണ്?,രൂപ,ഇന്ത്യയുടെ നാണയം പഞ്ചദാബദ്ധമായിരുന്നു.,False,ഇന്ത്യൻ രൂപയാണ്.,True
6,ഒരു വർഷത്തിൽ സാധാരണ എത്ര മാസങ്ങളുണ്ട്?,12,4 മാസങ്ങൾ.,False,ഏതാം മാസങ്ങൾക്കെതിരെ 12 മാസങ്ങൾക്കേതിരെ.,True
7,കേരളം ഇന്ത്യയുടെ ഏത് ഭാഗത്താണ്?,തെക്കുപടിഞ്ഞാറ്,കേരളം ഈസ്റ്റിയയുടെ കൊല്ലാശൈലി ഭാഗത്താണ്.,False,കേരളം ഇന്ത്യയുടെ തെക്കുപടിഞ്ഞാറൻ തീരത്താണ്.,False
8,വെള്ളത്തിന്റെ രാസസൂത്രം എന്താണ്?,H2O,വെള്ളത്തിന്റെ രാസസൂത്രം പുരുഷായിക സൂത്രമായിരിക...,False,H2O ആണ്.,True
9,സസ്യങ്ങൾ സൂര്യപ്രകാശം ഉപയോഗിച്ച് ആഹാരം നിർമ്മി...,പ്രകാശസംശ്ലേഷണം,വിദ്യാർത്ഥിക അനേകം സസ്യങ്ങളുടെ സൂര്യപ്രകാശം ഉപ...,False,സസ്യങ്ങൾ സൂര്യപ്രകാശം ഉപയോഗിച്ച് ആഹാരം നിർമ്മി...,False


In [ ]:
import re

def normalize(text):
    return re.sub(r"\s+", " ", str(text)).strip()

results["baseline_match"] = results.apply(
    lambda row: normalize(row["expected_answer"])
    in normalize(row["baseline_answer"]),
    axis=1
)

results["context_match"] = results.apply(
    lambda row: normalize(row["expected_answer"])
    in normalize(row["context_answer"]),
    axis=1
)

In [ ]:
summary = pd.DataFrame([
    {
        "condition": "baseline_no_context",
        "n": len(results),
        "exact_substring_match_rate": results["baseline_match"].mean(),
        "mean_latency_s": results["baseline_latency_s"].mean()
    },
    {
        "condition": "verified_context",
        "n": len(results),
        "exact_substring_match_rate": results["context_match"].mean(),
        "mean_latency_s": results["context_latency_s"].mean()
    }
])

print(summary)

             condition   n  exact_substring_match_rate  mean_latency_s
0  baseline_no_context  10                         0.0        2.833115
1     verified_context  10                         0.6        2.130639


In [ ]:
print(results[[
    "question",
    "expected_answer",
    "baseline_answer",
    "baseline_match",
    "context_answer",
    "context_match"
]].to_string(index=False))

                                                             question expected_answer                                                baseline_answer  baseline_match                                                   context_answer  context_match
                                         കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?   തിരുവനന്തപുരം          കേരളത്തിന്റെ തലസ്ഥാനം കൊടുപ്പായി 13 മീറ്റർ വൃത്തമാണ്.           False                         കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരം ആണ്.           True
                                           ഇന്ത്യയുടെ തലസ്ഥാനം ഏതാണ്?        ന്യൂഡൽഹി    ഇന്ത്യയുടെ തലസ്ഥാനം പഞ്ചകാലികളിൽ നിർമ്മിച്ച കോടിയായിരുന്നു.           False                                ഇന്ത്യയുടെ തലസ്ഥാനം ന്യൂഡൽഹി ആണ്.           True
                                     കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ ഏതാണ്?          മലയാളം കേരളത്തിന്റെ പല സമയങ്ങൾ കൊടുത്ത ഉദ്യോഗിക ഭാഷ എന്നിവയും അറിയപ്�           False                     കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ എന്നാൽ മലയാളം ആണ്.           True
                    

In [ ]:
print("Baseline match rate:", results["baseline_match"].mean())
print("Context match rate:", results["context_match"].mean())

Baseline match rate: 0.0
Context match rate: 0.6


In [ ]:
import pandas as pd

review = results[[
    "question",
    "expected_answer",
    "baseline_answer",
    "baseline_match",
    "context_answer",
    "context_match"
]].copy()

review["baseline_semantic_correct"] = ""
review["context_semantic_correct"] = ""
review["notes"] = ""

print(review.to_string(index=False))

                                                             question expected_answer                                                baseline_answer  baseline_match                                                   context_answer  context_match baseline_semantic_correct context_semantic_correct notes
                                         കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?   തിരുവനന്തപുരം          കേരളത്തിന്റെ തലസ്ഥാനം കൊടുപ്പായി 13 മീറ്റർ വൃത്തമാണ്.           False                         കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരം ആണ്.           True                                                         
                                           ഇന്ത്യയുടെ തലസ്ഥാനം ഏതാണ്?        ന്യൂഡൽഹി    ഇന്ത്യയുടെ തലസ്ഥാനം പഞ്ചകാലികളിൽ നിർമ്മിച്ച കോടിയായിരുന്നു.           False                                ഇന്ത്യയുടെ തലസ്ഥാനം ന്യൂഡൽഹി ആണ്.           True                                                         
                                     കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ ഏതാണ്?          മലയാളം കേരളത്തി

In [ ]:
for col in [
    "baseline_semantic_correct",
    "context_semantic_correct"
]:
    correct_rate = (
        review[col].str.strip().str.lower().eq("correct").mean()
    )
    print(f"{col}: {correct_rate:.1%}")

baseline_semantic_correct: 0.0%
context_semantic_correct: 0.0%


In [ ]:
print("Baseline mean latency:",
      results["baseline_latency_s"].mean())

print("Context mean latency:",
      results["context_latency_s"].mean())

Baseline mean latency: 2.8331152569000095
Context mean latency: 2.1306389655999736


In [ ]:
print(review["baseline_semantic_correct"].value_counts(dropna=False))
print(review["context_semantic_correct"].value_counts(dropna=False))

baseline_semantic_correct
    10
Name: count, dtype: int64
context_semantic_correct
    10
Name: count, dtype: int64


In [ ]:
for col in [
    "baseline_semantic_correct",
    "context_semantic_correct"
]:
    labels = review[col].str.strip().str.lower()
    assessable = labels.isin(["correct", "incorrect", "partial"])
    correct = labels.eq("correct")

    rate = correct[assessable].mean() if assessable.any() else float("nan")
    print(f"{col}: {rate:.1%} ({correct[assessable].sum()}/{assessable.sum()})")

baseline_semantic_correct: nan% (0/0)
context_semantic_correct: nan% (0/0)


In [ ]:
print(review[[
    "question",
    "expected_answer",
    "baseline_answer",
    "context_answer",
    "baseline_semantic_correct",
    "context_semantic_correct"
]].to_string(index=False))


                                                             question expected_answer                                                baseline_answer                                                   context_answer baseline_semantic_correct context_semantic_correct
                                         കേരളത്തിന്റെ തലസ്ഥാനം ഏതാണ്?   തിരുവനന്തപുരം          കേരളത്തിന്റെ തലസ്ഥാനം കൊടുപ്പായി 13 മീറ്റർ വൃത്തമാണ്.                         കേരളത്തിന്റെ തലസ്ഥാനം തിരുവനന്തപുരം ആണ്.                                                   
                                           ഇന്ത്യയുടെ തലസ്ഥാനം ഏതാണ്?        ന്യൂഡൽഹി    ഇന്ത്യയുടെ തലസ്ഥാനം പഞ്ചകാലികളിൽ നിർമ്മിച്ച കോടിയായിരുന്നു.                                ഇന്ത്യയുടെ തലസ്ഥാനം ന്യൂഡൽഹി ആണ്.                                                   
                                     കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ ഏതാണ്?          മലയാളം കേരളത്തിന്റെ പല സമയങ്ങൾ കൊടുത്ത ഉദ്യോഗിക ഭാഷ എന്നിവയും അറിയപ്�                     കേരളത്തിന്റെ ഔദ്യോഗിക ഭാഷ എന്നാൽ മലയ